# Pipeline de lead scoring — v2 (treinamento + produção)

Fluxo de dataframes (um `.copy()` em cada etapa, para voltar a qualquer estágio sem rodar tudo de novo):

`df_raw` → `df1` (fill NaN) → **`df2` (EDA)** → `df3` (feature eng + encoding) → `df4` (feature selection) → `df5` (padronização) → modelos

Na parte de **produção** os mesmos passos ganham o sufixo `_prod` e reaproveitam os artefatos salvos no treino (sem re-treinar, sem feature selection, sem EDA).


## 1. Importação de bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, roc_curve, confusion_matrix)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 120)
plt.rcParams["figure.figsize"] = (7, 4)
CAMINHO_MODELO = "modelo_vencedor.joblib"   # salvo na mesma pasta do notebook
print("Bibliotecas carregadas.")

## 2. Helper functions

Funções ficam **só** para avaliação/plots e para gerar dados sintéticos. As etapas de **fill NaN** e **feature engineering** ficam propositalmente *inline* (sem função), porque nesta fase de teste você vai mexer bastante nelas — depois de estabilizar, é só encapsular.


In [ ]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def gerar_base_sintetica(n=120_000, taxa_alvo=0.0023, random_state=RANDOM_STATE):
    """Gera df_raw com o mesmo schema da base real (34 colunas, sem 'idade').
    Injeta NaNs realistas (pref_*, persona, tipo_cartao, vencimento_fatura)."""
    rng = np.random.default_rng(random_state)
    ufs = ["SP","RJ","MG","RS","PR","BA","SC","PE","CE","GO","DF","ES","PA","MA","PB"]
    bandeiras = ["VISA","MASTERCARD","ELO","AMEX"]
    tipos = ["STANDARD","GOLD","PLATINUM","SIGNATURE","INFINITE"]
    personas = ["FAMILIA_CONSERVADORA","CONECTADO","JOVEM_URBANO","PREMIUM_VIAJANTE","PRATICO","INVESTIDOR"]
    prefs = ["RESTAURANTES","VIAGENS","SUPERMERCADO","COMBUSTIVEL","SAUDE","EDUCACAO","VESTUARIO","STREAMING"]
    produtos = [(101,"SEGURO_PROTECAO_PESSOAL"),(102,"SEGURO_RESIDENCIAL"),(103,"SEGURO_HOSPITALAR")]

    uf = rng.choice(ufs, size=n)
    idade = np.clip(rng.normal(45, 14, n), 18, 90).astype(int)   # interno: vira dt_nascimento
    tipo_cartao = rng.choice(tipos, size=n, p=[.45,.28,.15,.08,.04])
    persona = rng.choice(personas, size=n)
    fl_tel_1=rng.binomial(1,.97,n); fl_tel_2=rng.binomial(1,.55,n); fl_tel_3=rng.binomial(1,.30,n)
    fl_tel_4=rng.binomial(1,.15,n); fl_tel_5=rng.binomial(1,.07,n); fl_tel_pix=rng.binomial(1,.60,n)
    ac_pessoais=rng.binomial(1,.04,n); fl_cap=rng.binomial(1,.03,n); fl_hosp=rng.binomial(1,.05,n)
    fl_hl = rng.binomial(1,.20,n)
    venc = rng.integers(1,29,n).astype(float)

    premium = np.isin(tipo_cartao, ["PLATINUM","SIGNATURE","INFINITE"]).astype(int)
    persona_ef = pd.Series(persona).map({"FAMILIA_CONSERVADORA":.5,"PREMIUM_VIAJANTE":.4,"INVESTIDOR":.2,
                                          "PRATICO":0.,"CONECTADO":-.1,"JOVEM_URBANO":-.3}).values
    lp = (0.9*fl_hl + 1.3*ac_pessoais + 1.1*fl_cap + 1.0*fl_hosp + 0.4*premium
          + 0.015*(idade-45) + 0.3*fl_tel_pix + persona_ef + rng.normal(0,.5,n))
    lo, hi = -25., 10.
    for _ in range(80):
        mid=(lo+hi)/2
        if _sigmoid(lp+mid).mean() > taxa_alvo: hi=mid
        else: lo=mid
    prob = _sigmoid(lp + (lo+hi)/2)
    fl_venda = rng.binomial(1, prob)

    id_prod = np.full(n, np.nan, dtype=object); ds_prod = np.full(n, np.nan, dtype=object)
    iv = np.where(fl_venda==1)[0]; esc = rng.integers(0,len(produtos),iv.size)
    id_prod[iv]=[produtos[i][0] for i in esc]; ds_prod[iv]=[produtos[i][1] for i in esc]

    safra = (pd.Timestamp("2025-06-01") - pd.to_timedelta(rng.integers(0,78,n)*30, unit="D")).normalize()
    dt_nasc = (pd.Timestamp("2025-06-01") - pd.to_timedelta(idade*365, unit="D")).normalize()

    def pref(pnull):
        v = rng.choice(prefs, size=n).astype(object)
        v[rng.random(n) < pnull] = np.nan
        return v

    # injeta NaN em algumas colunas (missingness realista)
    persona = persona.astype(object);   persona[rng.random(n) < .05] = np.nan
    tipo_cartao = tipo_cartao.astype(object); tipo_cartao[rng.random(n) < .02] = np.nan
    venc[rng.random(n) < .03] = np.nan

    return pd.DataFrame({
        "ano_base":2025,"mes_base":6,
        "nr_cartao":rng.integers(10**9,10**10,n).astype(str),
        "nr_cpf":rng.integers(10**10,10**11,n).astype(str),
        "cidade":[u+"_CID_"+str(c) for u,c in zip(uf, rng.integers(0,400,n))],
        "uf":uf,"cep":rng.integers(10**6,10**8,n).astype(str),
        "fl_tel_1":fl_tel_1,"fl_tel_2":fl_tel_2,"fl_tel_3":fl_tel_3,
        "fl_tel_4":fl_tel_4,"fl_tel_5":fl_tel_5,"fl_tel_pix":fl_tel_pix,
        "bandeira_cartao":rng.choice(bandeiras,size=n,p=[.45,.40,.12,.03]),
        "tipo_cartao":tipo_cartao,"sexo":rng.choice(["M","F"],size=n),
        "vencimento_fatura":venc,"safra":safra,
        "pref_1":pref(.85),"pref_2":pref(.93),"pref_3":pref(.97),
        "ac_pessoais":ac_pessoais,"fl_cap":fl_cap,"fl_hosp":fl_hosp,
        "dt_nascimento":dt_nasc,"persona":persona,
        "aa_AnoCamp":2025,"mm_MesCamp":rng.integers(1,9,n),"fl_hl":fl_hl,
        "id_arquivo":rng.integers(1,50,n),"id_campanha":rng.integers(900,999,n),
        "id_produto_vendido":id_prod,"ds_produto_vendido":ds_prod,"fl_venda":fl_venda,
    })

In [ ]:
def precision_recall_lift_at_k(y_true, scores, frac):
    y_true = np.asarray(y_true)
    k = max(1, int(len(scores)*frac))
    idx = np.argsort(scores)[::-1][:k]
    tp = y_true[idx].sum()
    return tp/k, tp/y_true.sum(), (tp/k)/y_true.mean()

def metricas(nome, y_true, proba, frac=0.05):
    p, r, l = precision_recall_lift_at_k(y_true, proba, frac)
    pct = int(frac*100)
    return {"modelo":nome,
            "ROC_AUC": roc_auc_score(y_true, proba),
            "PR_AUC":  average_precision_score(y_true, proba),
            f"prec@{pct}%": p, f"rec@{pct}%": r, f"lift@{pct}%": l}

def plot_pr_roc(y_true, proba, nome=""):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    pr, rc, _ = precision_recall_curve(y_true, proba)
    ax[0].plot(rc, pr); ax[0].axhline(np.mean(y_true), ls="--", color="grey")
    ax[0].set(xlabel="Recall", ylabel="Precision",
              title=f"PR — {nome} (AP={average_precision_score(y_true, proba):.3f})")
    fpr, tpr, _ = roc_curve(y_true, proba)
    ax[1].plot(fpr, tpr); ax[1].plot([0,1],[0,1], ls="--", color="grey")
    ax[1].set(xlabel="FPR", ylabel="TPR", title=f"ROC — AUC={roc_auc_score(y_true, proba):.3f}")
    plt.tight_layout(); plt.show()

def plot_threshold_tradeoff(y_true, proba):
    """Precision/Recall/F1 e VOLUME de leads vs threshold. Retorna o threshold que maximiza F1."""
    y_true = np.asarray(y_true)
    ths = np.linspace(0.01, 0.99, 99)
    prec, rec, f1, vol = [], [], [], []
    for t in ths:
        pred = (proba >= t).astype(int); pp = pred.sum()
        tp = ((pred==1) & (y_true==1)).sum()
        p = tp/pp if pp else 0.0
        r = tp/y_true.sum()
        prec.append(p); rec.append(r); vol.append(pp)
        f1.append(2*p*r/(p+r) if (p+r) else 0.0)
    prec, rec, f1, vol = map(np.array, (prec, rec, f1, vol))
    t_best = ths[int(np.argmax(f1))]
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.plot(ths, prec, label="Precision"); ax1.plot(ths, rec, label="Recall"); ax1.plot(ths, f1, label="F1")
    ax1.axvline(t_best, ls="--", color="red", label=f"F1 max @ {t_best:.2f}")
    ax1.set_xlabel("Threshold"); ax1.set_ylabel("Precision / Recall / F1"); ax1.legend(loc="upper right")
    ax2 = ax1.twinx()
    ax2.plot(ths, vol, color="grey", alpha=0.35)
    ax2.set_ylabel("Volume de leads (nº de ligações)", color="grey")
    plt.title("Trade-off de threshold: métrica vs. volume de leads")
    plt.tight_layout(); plt.show()
    return t_best

def tabela_capacidade(y_true, proba, fracs=(0.005, 0.01, 0.02, 0.05, 0.10, 0.20)):
    """Para cada fatia do topo da lista: quantos ligar, precision, recall, lift e conversões esperadas."""
    y_true = np.asarray(y_true); linhas = []
    for f in fracs:
        p, r, l = precision_recall_lift_at_k(y_true, proba, f)
        n = int(len(proba)*f)
        linhas.append({"top_%": f*100, "n_leads": n, "precision": round(p,4),
                       "recall": round(r,4), "lift": round(l,2),
                       "conversoes_esperadas": int(round(p*n))})
    return pd.DataFrame(linhas)

def plot_matriz_confusao(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y_true, pred)
    fig, ax = plt.subplots(figsize=(4.3, 3.9))
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v:,}", ha="center", va="center")
    ax.set(xticks=[0,1], yticks=[0,1], xticklabels=["nao","sim"], yticklabels=["nao","sim"],
           xlabel="Previsto", ylabel="Real", title=f"Matriz de confusao @ thr={thr:.2f}")
    plt.tight_layout(); plt.show()

print("Helpers definidos.")

## 3. Carregamento de dados — `df_raw`

In [ ]:
# Base sintetica (roda de ponta a ponta). Para a base real, comente a linha abaixo
# e use o read_. O contrato: df_raw precisa ter as mesmas colunas.
df_raw = gerar_base_sintetica(n=120_000)
# df_raw = pd.read_parquet("caminho/sua_base.parquet")

print("Shape:", df_raw.shape, "| conversao:", round(df_raw["fl_venda"].mean(), 5))
df_raw.head(3)

## 4. Fill NaN — `df1`

**Sobre a sua ideia de "priorizar o preenchimento com a variável resposta positiva":** cuidado — calcular o valor de imputação *olhando o alvo* (ex.: preencher a idade dos compradores com a média dos compradores) injeta o `fl_venda` dentro da feature. Isso é **target leakage**: infla as métricas de forma irreal e é **impossível de reproduzir em produção**, onde você não tem a resposta. A intuição boa por trás da ideia é *não perder positivos*, já que eles são raríssimos. A forma correta de honrar isso:

1. **Nunca dropar linhas por NaN** — sempre preencher (dropar poderia jogar fora positivos preciosos).
2. **A ausência pode ser sinal.** Para **categóricas**, preenchemos com a categoria `"DESCONHECIDO"` — quando fizermos one-hot, ela vira uma coluna própria e o modelo aprende se "faltar" ajuda a prever. Para **numéricas**, a mediana esconde a ausência, então criamos uma **flag `_faltando`** explícita.
3. **Estatísticas de imputação calculadas SEM olhar `y`** (mediana/moda gerais) e **guardadas** (`PARAMS_FILL`) para reaplicar iguais em produção — evita descasamento treino/produção.


In [ ]:
df1 = df_raw.copy()
alvo = "fl_venda"

# categoricas -> "DESCONHECIDO" (a propria categoria preserva o sinal de ausencia)
cat_fill = ["uf","bandeira_cartao","tipo_cartao","sexo","persona","pref_1","pref_2","pref_3"]
for c in cat_fill:
    df1[c] = df1[c].fillna("DESCONHECIDO")

# numericas -> flag de ausencia + mediana (guardada para producao)
PARAMS_FILL = {}
num_fill = ["vencimento_fatura"]
for c in num_fill:
    df1[c + "_faltando"] = df1[c].isna().astype(int)
    PARAMS_FILL[c] = float(df1[c].median())
    df1[c] = df1[c].fillna(PARAMS_FILL[c])

# id/ds_produto_vendido continuam NaN de proposito (sao leakage e serao dropados no df3)
_check = df1.drop(columns=["id_produto_vendido","ds_produto_vendido"])
print("NaNs restantes (fora das colunas de venda):", int(_check.isna().sum().sum()),
      "| PARAMS_FILL:", PARAMS_FILL)

## (`df2`) EDA — reservado

`df2` é o snapshot para exploração. Mantido mínimo aqui; expanda à vontade. **Não** é usado no fluxo de modelagem, e é pulado na produção.


In [ ]:
df2 = df1.copy()
print("Conversao:", round(df2[alvo].mean(), 5))
print(df2["persona"].value_counts(dropna=False).head())

## 5. Feature engineering + encoding — `df3`

**Estratégias:**
- **`idade`** = (data-base − `dt_nascimento`) em anos. A base não traz idade pronta, só a data de nascimento.
- **`tenure_meses`** = meses desde `safra` (emissão do cartão). Cliente novo x antigo se comporta diferente — e `fl_hl` (hotlist) é justamente cartão recente.
- **`qtd_telefones`** = soma dos `fl_tel_*` — contatabilidade, pré-condição para vender no telemarketing.
- **`tem_pref_1`** = existe preferência? Como `pref_*` é quase toda nula, o *fato de existir* costuma valer mais que o valor.
- **Descartes:** `id_produto_vendido`/`ds_produto_vendido` (**leakage** — só existem após a venda); identificadores (`nr_cartao`, `nr_cpf`, `cep`, `cidade`, `id_*`); e colunas já derivadas/constantes.

**Encoding — por que one-hot:** as categóricas restantes (`uf`, `bandeira_cartao`, `tipo_cartao`, `sexo`, `persona`) são **nominais** (sem ordem). Codificá-las como número (0,1,2,...) inventaria uma ordem falsa que modelos lineares leem como grandeza — logo usamos **one-hot** (uma coluna 0/1 por categoria). A cardinalidade aqui é baixa, então não explode a dimensionalidade. Em produção, alinhamos as colunas ao conjunto do treino (`COLS_ENG`).


In [ ]:
df3 = df1.copy()
tel_cols = ["fl_tel_1","fl_tel_2","fl_tel_3","fl_tel_4","fl_tel_5","fl_tel_pix"]
base = pd.to_datetime(dict(year=df3["ano_base"], month=df3["mes_base"], day=1))

df3["idade"]        = ((base - pd.to_datetime(df3["dt_nascimento"])).dt.days / 365.25).astype(int)
df3["tenure_meses"] = ((base - pd.to_datetime(df3["safra"])).dt.days / 30.0).clip(lower=0)
df3["qtd_telefones"] = df3[tel_cols].sum(axis=1)
df3["tem_pref_1"]   = (df3["pref_1"] != "DESCONHECIDO").astype(int)

cat_ohe = ["uf","bandeira_cartao","tipo_cartao","sexo","persona"]
drop_cols = ["id_produto_vendido","ds_produto_vendido",              # leakage
             "nr_cartao","nr_cpf","cep","cidade","id_arquivo","id_campanha",  # identificadores
             "dt_nascimento","safra","ano_base","mes_base","aa_AnoCamp",      # derivadas/constantes
             "pref_1","pref_2","pref_3"]                              # viraram flag
df3 = df3.drop(columns=drop_cols)
df3 = pd.get_dummies(df3, columns=cat_ohe, dtype=int)

COLS_ENG = [c for c in df3.columns if c != alvo]   # colunas apos encoding (para alinhar producao)
print("df3:", df3.shape, "| features apos encoding:", len(COLS_ENG))

## 6. Feature selection — `df4`

Menos features = menos ruído, menos overfitting e treino mais rápido. Duas regras simples e sem olhar o alvo:
1. **Variância quase-zero** — colunas praticamente constantes não separam nada.
2. **Correlação alta (>0,95)** — features redundantes; mantemos só uma de cada par.

A lista final `FEATURES_SEL` é salva e **reaproveitada em produção** (lá não recalculamos seleção).


In [ ]:
df4 = df3.copy()
X_cols = [c for c in df4.columns if c != alvo]

# 1) variancia quase-zero
var = df4[X_cols].var()
baixa_var = var[var < 1e-4].index.tolist()

# 2) correlacao alta
corr = df4[X_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
alta_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]

remover = sorted(set(baixa_var + alta_corr))
FEATURES_SEL = [c for c in X_cols if c not in remover]
df4 = df4[FEATURES_SEL + [alvo]]
print(f"Removidas {len(remover)} | restaram {len(FEATURES_SEL)} features.")
print("Removidas (amostra):", remover[:8])

## 7. Padronização — `df5`

Padronizamos as **numéricas contínuas** (média 0, desvio 1). Isso é essencial para modelos sensíveis a escala (Regressão Logística) e inofensivo para árvores. O `StandardScaler` é **ajustado só no treino** e aplicado no teste — se ajustar no conjunto todo, vaza informação do teste. As colunas 0/1 (flags e one-hot) não precisam de escala.


In [ ]:
df5 = df4.copy()
X = df5[FEATURES_SEL].copy()
y = df5[alvo].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

NUM_CONT = [c for c in ["idade","tenure_meses","qtd_telefones","vencimento_fatura","mm_MesCamp"]
            if c in FEATURES_SEL]
scaler = StandardScaler().fit(X_train[NUM_CONT])
X_train = X_train.copy(); X_test = X_test.copy()
X_train[NUM_CONT] = scaler.transform(X_train[NUM_CONT])
X_test[NUM_CONT]  = scaler.transform(X_test[NUM_CONT])
print("Treino:", X_train.shape, "| Teste:", X_test.shape, "| positivos treino:", int(y_train.sum()))

## 8. Treinamento dos modelos (loop)

Treinamos vários modelos no mesmo pré-processamento para comparar de forma justa. Para o desbalanceamento usamos `class_weight="balanced"` (LogReg, árvores) ou `sample_weight` balanceado (HistGradientBoosting, que não tem `class_weight`).


In [ ]:
sw = compute_sample_weight(class_weight="balanced", y=y_train)

modelos = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample",
                                           n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                       l2_regularization=1.0, random_state=RANDOM_STATE),
}

ajustados = {}
for nome, mdl in modelos.items():
    if nome == "HistGradBoosting":
        mdl.fit(X_train, y_train, sample_weight=sw)
    else:
        mdl.fit(X_train, y_train)
    ajustados[nome] = mdl
    print("treinado:", nome)

## 9. Benchmark dos modelos (holdout)

Avaliação no conjunto de teste. **PR-AUC** é a métrica principal em base rara; ROC-AUC é reportado mas otimista. `prec@5%`, `rec@5%` e `lift@5%` são a leitura operacional (o que acontece se ligarmos para os 5% mais bem pontuados).


In [ ]:
probas = {}
linhas = []
for nome, mdl in ajustados.items():
    p = mdl.predict_proba(X_test)[:, 1]
    probas[nome] = p
    linhas.append(metricas(nome, y_test, p))

bench = pd.DataFrame(linhas).set_index("modelo").sort_values("PR_AUC", ascending=False)
print("Taxa-base (teste):", round(float(y_test.mean()), 5))
bench.round(4)

## 10. Cross-validação dos modelos

Um único split pode dar sorte/azar. A validação cruzada **StratifiedKFold** (estratificada para manter a proporção de positivos em cada *fold*) mede a **robustez**: média e desvio-padrão da PR-AUC em 5 partições. Usamos `scoring="average_precision"` (= PR-AUC), que é de ranking e razoavelmente robusto ao balanceamento na CV.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
linhas_cv = []
for nome, mdl in modelos.items():
    scores = cross_val_score(mdl, X_train, y_train, cv=skf,
                             scoring="average_precision", n_jobs=-1)
    linhas_cv.append({"modelo": nome,
                      "PR_AUC_cv_media": scores.mean(),
                      "PR_AUC_cv_std": scores.std()})
    print(f"{nome:18s} PR-AUC(cv) = {scores.mean():.4f} +/- {scores.std():.4f}")

## 11. Benchmark (cross-validação) e escolha do vencedor

Escolhemos o vencedor pela **média da PR-AUC na CV** (mais confiável que um único holdout). Empates ou casos-limite: prefira o modelo mais simples/estável.


In [ ]:
bench_cv = pd.DataFrame(linhas_cv).set_index("modelo").sort_values("PR_AUC_cv_media", ascending=False)
print(bench_cv.round(4))

nome_vencedor = bench_cv["PR_AUC_cv_media"].idxmax()
modelo_vencedor = ajustados[nome_vencedor]
proba_venc = probas[nome_vencedor]
print("\nModelo vencedor:", nome_vencedor)

## 12. Avaliação do modelo vencedor

Curvas PR e ROC; depois o **trade-off de threshold** e a matriz de confusão.


In [ ]:
plot_pr_roc(y_test, proba_venc, nome_vencedor)

### Escolha do threshold — métrica vs. volume de leads

O threshold que **maximiza o F1** (marcado em vermelho) equilibra precision e recall, mas pode gerar **poucos leads**. Como você pode ter mais atendentes para discar mais, dá para **abaixar o threshold**: sobe o volume e o recall, cai a precision. Use este gráfico + a tabela de capacidade para casar o corte com o tamanho do seu pacote de ligações.


In [ ]:
THRESHOLD_F1 = plot_threshold_tradeoff(y_test, proba_venc)
print("Threshold que maximiza F1:", round(float(THRESHOLD_F1), 3))

In [ ]:
# Planejamento por capacidade: se ligarmos para os top-k%, o que acontece?
tabela_capacidade(y_test, proba_venc)

In [ ]:
# Escolha o corte pela capacidade da operacao. Comece pelo F1 e ajuste se quiser mais volume.
THRESHOLD_ESCOLHIDO = float(THRESHOLD_F1)   # <-- edite aqui (ex.: 0.30) para gerar mais leads
plot_matriz_confusao(y_test, proba_venc, THRESHOLD_ESCOLHIDO)

## 13. Salvar o modelo vencedor + artefatos

Salvamos tudo que a produção precisa para reproduzir o pré-processamento **idêntico**: o modelo, o `scaler`, os parâmetros de imputação, as listas de colunas e o threshold escolhido.


In [ ]:
bundle = {
    "nome_vencedor": nome_vencedor,
    "modelo": modelo_vencedor,
    "scaler": scaler,
    "PARAMS_FILL": PARAMS_FILL,
    "cat_fill": cat_fill, "num_fill": num_fill,
    "tel_cols": tel_cols, "cat_ohe": cat_ohe, "drop_cols": drop_cols,
    "COLS_ENG": COLS_ENG, "FEATURES_SEL": FEATURES_SEL, "NUM_CONT": NUM_CONT,
    "threshold": THRESHOLD_ESCOLHIDO,
}
joblib.dump(bundle, CAMINHO_MODELO)
print("Modelo e artefatos salvos em:", CAMINHO_MODELO)

# Produção (ou teste)

Aqui entram dados que **não** participaram do treino/validação. Repetimos **apenas** o necessário — fill NaN e feature engineering — reaproveitando os artefatos salvos. **Sem** treino, **sem** feature selection, **sem** EDA. Sufixo `_prod` em tudo.

Dois modos:
- **Produção:** dados sem resposta → gerar a lista priorizada de leads.
- **Teste:** dados com `fl_venda` conhecido → também avaliar as métricas.


## P1. Carregar artefatos + dados novos — `df_raw_prod`

In [ ]:
bundle = joblib.load(CAMINHO_MODELO)
modelo_vencedor = bundle["modelo"];        scaler = bundle["scaler"]
PARAMS_FILL = bundle["PARAMS_FILL"]
cat_fill = bundle["cat_fill"];             num_fill = bundle["num_fill"]
tel_cols = bundle["tel_cols"];             cat_ohe = bundle["cat_ohe"]
drop_cols = bundle["drop_cols"];           COLS_ENG = bundle["COLS_ENG"]
FEATURES_SEL = bundle["FEATURES_SEL"];     NUM_CONT = bundle["NUM_CONT"]
THRESHOLD = bundle["threshold"]

# Dados NOVOS (outra semente). Para dados reais: df_raw_prod = pd.read_parquet(...)
df_raw_prod = gerar_base_sintetica(n=60_000, random_state=2024)

tem_resposta = "fl_venda" in df_raw_prod.columns
y_prod = df_raw_prod["fl_venda"].astype(int) if tem_resposta else None
print("Shape:", df_raw_prod.shape, "| tem resposta (modo teste):", tem_resposta)

## P2. Fill NaN — `df1_prod` (reaproveita `PARAMS_FILL`)

In [ ]:
df1_prod = df_raw_prod.copy()
for c in cat_fill:
    df1_prod[c] = df1_prod[c].fillna("DESCONHECIDO")
for c in num_fill:
    df1_prod[c + "_faltando"] = df1_prod[c].isna().astype(int)
    df1_prod[c] = df1_prod[c].fillna(PARAMS_FILL[c])   # MESMA mediana do treino
_chk = df1_prod.drop(columns=["id_produto_vendido","ds_produto_vendido"], errors="ignore")
print("NaNs restantes (fora das colunas de venda):", int(_chk.isna().sum().sum()))

## P3. Feature engineering + encoding — `df3_prod` (alinha a `COLS_ENG`)

In [ ]:
df3_prod = df1_prod.copy()
base = pd.to_datetime(dict(year=df3_prod["ano_base"], month=df3_prod["mes_base"], day=1))
df3_prod["idade"]        = ((base - pd.to_datetime(df3_prod["dt_nascimento"])).dt.days / 365.25).astype(int)
df3_prod["tenure_meses"] = ((base - pd.to_datetime(df3_prod["safra"])).dt.days / 30.0).clip(lower=0)
df3_prod["qtd_telefones"] = df3_prod[tel_cols].sum(axis=1)
df3_prod["tem_pref_1"]   = (df3_prod["pref_1"] != "DESCONHECIDO").astype(int)

df3_prod = df3_prod.drop(columns=drop_cols)
df3_prod = pd.get_dummies(df3_prod, columns=cat_ohe, dtype=int)
# alinha exatamente as colunas do treino: cria faltantes=0, descarta categorias novas
df3_prod = df3_prod.reindex(columns=COLS_ENG, fill_value=0)
print("df3_prod alinhado:", df3_prod.shape, "| colunas == treino:", list(df3_prod.columns) == COLS_ENG)

## P4. Padronização — `df5_prod` (sem feature selection: só reaplica `FEATURES_SEL` e o `scaler`)

In [ ]:
df5_prod = df3_prod.copy()
X_prod = df5_prod[FEATURES_SEL].copy()
X_prod[NUM_CONT] = scaler.transform(X_prod[NUM_CONT])   # MESMO scaler do treino
print("X_prod:", X_prod.shape, "| NaNs:", int(X_prod.isna().sum().sum()))

## P5. Geração de leads (scoring)

In [ ]:
proba_prod = modelo_vencedor.predict_proba(X_prod)[:, 1]

leads = df_raw_prod[["nr_cpf", "uf", "fl_hl"]].copy()
leads["score"] = proba_prod
leads["ligar"] = (proba_prod >= THRESHOLD).astype(int)
leads = leads.sort_values("score", ascending=False).reset_index(drop=True)

leads.to_csv("leads_priorizados.csv", index=False)
print(f"Leads acima do threshold ({THRESHOLD:.2f}):",
      int(leads["ligar"].sum()), "de", len(leads))
leads.head(10)

## P6. Avaliação (modo teste — só se houver resposta)

In [ ]:
if tem_resposta:
    print(pd.Series(metricas("PRODUCAO", y_prod, proba_prod)).round(4).to_string())
    plot_pr_roc(y_prod, proba_prod, "Producao")
    print(tabela_capacidade(y_prod, proba_prod).to_string(index=False))
    plot_matriz_confusao(y_prod, proba_prod, THRESHOLD)
else:
    print("Sem coluna fl_venda: modo producao (apenas scoring). Nada a avaliar.")